# DS4SE 2026 — Week 3 (Group 1)
## ARC-style semantic clustering — Apache Tika *Detect & Parser*

`combined = ALPHA · structural + (1-ALPHA) · semantic` → distance → `AgglomerativeClustering`.

**Setup:** GPU runtime · add `HF_TOKEN` under Secrets · drag **`colab_bundle.zip`** into
`/content/` (cell 2 unpacks everything).

Embedding model (Group 1): **`nomic-ai/nomic-embed-code`**.

### 0 — Install

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes "scikit-learn<1.9" scipy matplotlib

### 1 — Imports

In [2]:
import os, re, zipfile, subprocess
from pathlib import Path
from collections import defaultdict
import numpy as np, torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
from google.colab import userdata

### 2 — Unpack the bundle & resolve paths
Drag only `colab_bundle.zip` into `/content/`; this cell unpacks it and the nested
`tika_sources.zip`.

In [ ]:
CONTENT = Path('/content')
bundle = CONTENT / 'colab_bundle.zip'
if bundle.is_file():
    zipfile.ZipFile(bundle).extractall(CONTENT)

SRC_ZIP = CONTENT / 'tika_sources.zip'
SRC_DIR = CONTENT / 'tika_sources'
if SRC_ZIP.is_file() and not SRC_DIR.exists():
    zipfile.ZipFile(SRC_ZIP).extractall(SRC_DIR)

RSF_PATH = CONTENT / 'tika_detect_parser.rsf'
print('sources:', sum(1 for _ in SRC_DIR.rglob('*.java')), 'java files')
print('rsf:', RSF_PATH.is_file())

### 3 — HF token

In [4]:
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN under the Secrets (key) tab.'

### 4 — 4-bit NF4 config
nomic-embed-code is a 7B model; NF4 keeps it inside a T4's 16 GB.

In [5]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

### 5 — Load `nomic-embed-code`

In [6]:
EMB_MODEL = 'nomic-ai/nomic-embed-code'
tok = AutoTokenizer.from_pretrained(EMB_MODEL, token=HF_TOKEN, trust_remote_code=True)
emb = AutoModel.from_pretrained(
    EMB_MODEL, token=HF_TOKEN, trust_remote_code=True,
    quantization_config=bnb, device_map='auto')
emb.eval()
print('embedding model loaded')

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/25.7k [00:00<?, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


embedding model loaded


### 6 — Parse the W1 filtered RSF
`deps[a]` = set of classes `a` depends on; `all_classes` = every FQN that appears.

In [7]:
deps = defaultdict(set)
all_classes = set()
for line in RSF_PATH.read_text(encoding='utf-8').splitlines():
    t = line.split()
    if len(t) >= 3 and t[0] == 'depends':
        deps[t[1]].add(t[2]); all_classes.update([t[1], t[2]])
print(len(all_classes), 'unique classes,', sum(len(v) for v in deps.values()), 'edges')

157 unique classes, 550 edges


### 7 — Resolve each FQN to its `.java` file
Inner classes (`Foo$Bar`) collapse onto their outer file. `files_list` / `file_fqns`
stay index-aligned — this ordering drives every matrix below.

In [8]:
def fqn_to_path(fqn):
    return SRC_DIR / (fqn.split('$', 1)[0].replace('.', '/') + '.java')

files_list, file_fqns, missing = [], [], []
for fqn in sorted(all_classes):
    p = fqn_to_path(fqn)
    if p.is_file():
        files_list.append(p); file_fqns.append(fqn)
    else:
        missing.append(fqn)
N = len(file_fqns)
idx = {fqn: i for i, fqn in enumerate(file_fqns)}
print(f'resolved {N}/{len(all_classes)} ; missing {len(missing)}:', missing[:5])

resolved 156/157 ; missing 1: ['org.apache.tika.config.TikaComponent']


### 8 — Embed each file
Mean-pool the last hidden state with the attention mask, then L2-normalize. nomic uses a
task prefix; code files routinely exceed 1000 tokens so `max_length=2048` (nomic's training length), not CodeBERT's 512.

In [9]:
PREFIX = 'search_document: '

@torch.inference_mode()
def embed_batch(texts, max_length=2048):
    enc = tok([PREFIX + t for t in texts], padding=True, truncation=True,
              max_length=max_length, return_tensors='pt').to(emb.device)
    out = emb(**enc).last_hidden_state            # (B, T, H)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return torch.nn.functional.normalize(pooled, p=2, dim=1).float().cpu().numpy()

vecs, B = [], 8
for i in range(0, N, B):
    texts = [p.read_text(encoding='utf-8', errors='ignore') for p in files_list[i:i+B]]
    vecs.append(embed_batch(texts))
    print(f'  embedded {min(i+B, N)}/{N}', end='\r')
embeddings = np.vstack(vecs)
print('\nembeddings:', embeddings.shape)

  embedded 156/156
embeddings: (156, 3584)


### 9 — Semantic similarity matrix
Embeddings are L2-normalized, so cosine = dot product. Clamp to [0, 1].

In [10]:
semantic_matrix = embeddings @ embeddings.T
semantic_matrix = np.clip(semantic_matrix, 0.0, 1.0)
np.fill_diagonal(semantic_matrix, 1.0)
print('semantic:', semantic_matrix.shape, 'mean', semantic_matrix.mean().round(3))

semantic: (156, 156) mean 0.641


### 10 — Structural similarity (raw)
Two signals: a **direct** dependency either way, plus the count of **shared neighbours**
(classes both depend on / are depended on by) — the structural analogue of co-citation.

In [11]:
neigh = {f: deps.get(f, set()) | {a for a in deps if f in deps[a]} for f in file_fqns}
struct_raw = np.zeros((N, N))
for i, a in enumerate(file_fqns):
    for j, b in enumerate(file_fqns):
        if i == j:
            continue
        direct = (b in deps.get(a, set())) or (a in deps.get(b, set()))
        shared = len(neigh[a] & neigh[b])
        struct_raw[i, j] = shared + (1 if direct else 0)
print('struct_raw max', struct_raw.max(), 'nnz', (struct_raw > 0).sum())

struct_raw max 43.0 nnz 11508


### 11 — Normalize structural matrix to [0, 1]

In [12]:
m = struct_raw.max()
struct_matrix = struct_raw / m if m > 0 else struct_raw
np.fill_diagonal(struct_matrix, 1.0)

### 12 — Combine → distance
`ALPHA=0.5` weights structure and semantics equally. Sweep 0.3 / 0.7 for the writeup.

In [13]:
ALPHA = 0.5
combined_sim = ALPHA * struct_matrix + (1 - ALPHA) * semantic_matrix
distance = 1.0 - combined_sim
np.fill_diagonal(distance, 0.0)
distance = np.clip(distance, 0.0, None)

### 13 — Agglomerative clustering for k = 5, 10, 20, 30, 40, 50
Run complete-linkage clustering for each target cluster count.

In [ ]:
K_VALUES = [5, 10, 20, 30, 40, 50]
arc_labels = {}
for k in K_VALUES:
    agg = AgglomerativeClustering(
        n_clusters=k, metric='precomputed', linkage='complete')
    arc_labels[k] = agg.fit_predict(distance)
    print(f'k={k}: {len(set(arc_labels[k]))} clusters')

### 14 — Write each ARC clustering as a separate RSF
One file per k value: `tika_arc_alpha0.5_k<k>.rsf`

In [ ]:
for k, labels in arc_labels.items():
    rsf_path = CONTENT / f'tika_arc_alpha{ALPHA}_k{k}.rsf'
    with open(rsf_path, 'w', encoding='utf-8') as f:
        for fqn, lab in zip(file_fqns, labels):
            f.write(f'contain cluster_{lab} {fqn}\n')
    print(f'wrote {rsf_path.name}')

In [ ]:
from collections import Counter

# Cluster size bar charts for each k
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, k in zip(axes.flat, K_VALUES):
    sizes = sorted(Counter(arc_labels[k]).values(), reverse=True)
    ax.bar(range(1, len(sizes)+1), sizes,
           color=['#d9534f' if s == 1 else '#337ab7' for s in sizes])
    ax.set_title(f'k={k}')
    ax.set_xlabel('cluster rank'); ax.set_ylabel('classes'); ax.grid(axis='y', alpha=.3)
fig.suptitle(f'ARC cluster sizes (alpha={ALPHA})', fontsize=13)
plt.tight_layout(); plt.savefig('/content/cluster_sizes_all_k.png', dpi=120); plt.show()

# Dendrogram (shared — doesn't depend on k)
Z = linkage(distance[np.triu_indices(N, 1)], method='complete')
plt.figure(figsize=(12, 4)); dendrogram(Z, no_labels=True, color_threshold=0.7)
plt.title('ARC dendrogram (complete linkage)')
plt.tight_layout(); plt.savefig('/content/dendrogram.png', dpi=120); plt.show()

# Similarity heatmap for each k
fig, axes = plt.subplots(2, 3, figsize=(15, 12))
for ax, k in zip(axes.flat, K_VALUES):
    order = np.argsort(arc_labels[k])
    im = ax.imshow(combined_sim[np.ix_(order, order)], cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'k={k}'); ax.axis('off')
fig.suptitle(f'Combined similarity heatmap by cluster (alpha={ALPHA})', fontsize=13)
plt.colorbar(im, ax=axes, fraction=0.02)
plt.savefig('/content/similarity_heatmap_all_k.png', dpi=120); plt.show()

---
**Deliverables:** download the six RSF files (`tika_arc_alpha0.5_k5.rsf` … `tika_arc_alpha0.5_k50.rsf`) and the three PNGs (`cluster_sizes_all_k.png`, `dendrogram.png`, `similarity_heatmap_all_k.png`); export the notebook to PDF.